In [1]:
!pip install databento

In [3]:
import databento as db

In [4]:
import databento as db
from databento import DBNStore, Dataset, Schema

import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get API key from environment
DB_API_KEY = os.getenv('DATABENTO_API_KEY')
print('Loaded API key:', '***' if DB_API_KEY else 'NOT FOUND')

# Initialize client
client = db.Historical(DB_API_KEY)

Loaded API key: ***


In [5]:
data = client.timeseries.get_range(
    dataset="XNAS.ITCH",
    symbols=["SPY"],
    schema="ohlcv-1m",
    start="2019-01-01T00:00:00",
    end="2025-07-10T00:00:00"
)

C:\Users\Tom\AppData\Local\Temp\ipykernel_12068\1827881145.py:1: BentoWarning: The streaming request contained one or more days which have reduced quality: 2021-07-07 (degraded), 2021-10-26 (degraded), 2022-09-19 (degraded). See: https://databento.com/docs/api-reference-historical/metadata/metadata-get-dataset-condition
  data = client.timeseries.get_range(


In [ ]:
data.to_file('spy_ohlcv_20190102_20250710.dbn')

<DBNStore(schema=ohlcv-1m)>

In [7]:
data.to_df().head()

,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
ts_event,,,,,,,,,
2019-01-02 09:00:00+00:00,33,2,7294,245.38,245.43,245.08,245.08,1655,SPY
2019-01-02 09:01:00+00:00,33,2,7294,245.01,245.19,245.01,245.19,7055,SPY
2019-01-02 09:02:00+00:00,33,2,7294,245.19,245.24,244.96,245.24,964,SPY
2019-01-02 09:03:00+00:00,33,2,7294,245.22,245.22,245.22,245.22,8,SPY
2019-01-02 09:04:00+00:00,33,2,7294,245.26,245.40,245.26,245.40,389,SPY


In [ ]:
# Save the DBNStore object to a .dbn file


In [27]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from datetime import datetime, time
import pytz

def plot_interactive_ohlc(df, start_date, end_date=None, or_start_time="09:30:00", or_end_time="10:15:00"):
    """
    Create an interactive OHLC plot with opening range analysis.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with OHLC data (columns: open, high, low, close, volume)
    start_date : str
        Start date in 'YYYY-MM-DD' format
    end_date : str, optional
        End date in 'YYYY-MM-DD' format. If None, uses only start_date
    or_start_time : str
        Opening range start time in 'HH:MM:SS' format (default: "09:30:00")
    or_end_time : str
        Opening range end time in 'HH:MM:SS' format (default: "10:15:00")
    
    Returns:
    --------
    plotly.graph_objects.Figure
        Interactive plotly figure
    """
    
    # Filter data by date range
    if end_date is None:
        # Single day
        filtered_df = df.loc[start_date]
        title_date = start_date
    else:
        # Date range
        filtered_df = df.loc[start_date:end_date]
        title_date = f"{start_date} to {end_date}"
    
    if filtered_df.empty:
        print(f"No data found for the specified date range: {title_date}")
        return None
    
    # Create subplots with secondary y-axis for volume
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        subplot_titles=(f'SPY OHLC - {title_date}', 'Volume'),
        row_heights=[0.7, 0.3]
    )
    
    # Create hover text for candlestick
    hover_text = []
    for i, row in filtered_df.iterrows():
        hover_text.append(
            f"Time: {i}<br>" +
            f"Open: ${row['open']:.2f}<br>" +
            f"High: ${row['high']:.2f}<br>" +
            f"Low: ${row['low']:.2f}<br>" +
            f"Close: ${row['close']:.2f}"
        )
    
    # Add candlestick chart
    fig.add_trace(
        go.Candlestick(
            x=filtered_df.index,
            open=filtered_df['open'],
            high=filtered_df['high'],
            low=filtered_df['low'],
            close=filtered_df['close'],
            name='SPY',
            text=hover_text,
            hoverinfo='text'
        ),
        row=1, col=1
    )
    
    # Add volume bars
    fig.add_trace(
        go.Bar(
            x=filtered_df.index,
            y=filtered_df['volume'],
            name='Volume',
            marker_color='rgba(158,158,158,0.8)',
            hovertemplate='<b>%{x}</b><br>' +
                         'Volume: %{y:,}<br>' +
                         '<extra></extra>'
        ),
        row=2, col=1
    )
    
    # Calculate Opening Range for each day in the dataset
    if end_date is None:
        # Single day analysis
        days_to_analyze = [start_date]
    else:
        # Multi-day analysis - get unique dates
        days_to_analyze = filtered_df.index.date
        days_to_analyze = sorted(list(set(days_to_analyze)))
        days_to_analyze = [str(day) for day in days_to_analyze]
    
    # Add OR analysis for each day
    colors = ['red', 'green', 'blue', 'orange', 'purple']  # Cycle through colors for multiple days
    
    for i, day in enumerate(days_to_analyze):
        try:
            # Get day data
            day_data = df.loc[day]
            if day_data.empty:
                continue
                
            # Create OR time range
            timezone = day_data.index.tz if hasattr(day_data.index, 'tz') else None
            or_start = pd.Timestamp(f"{day} {or_start_time}", tz=timezone)
            or_end = pd.Timestamp(f"{day} {or_end_time}", tz=timezone)
            
            # Get OR data
            or_data = day_data.loc[or_start:or_end]
            if or_data.empty:
                continue
                
            or_high = or_data['high'].max()
            or_low = or_data['low'].min()
            
            # Choose colors
            high_color = colors[i % len(colors)]
            low_color = colors[(i + 1) % len(colors)]
            
            # Add horizontal lines for OR high and low
            fig.add_hline(
                y=or_high,
                line_dash="dash",
                line_color=high_color,
                line_width=2,
                annotation_text=f"OR High {day}: ${or_high:.2f}",
                annotation_position="top right",
                row=1
            )
            
            fig.add_hline(
                y=or_low,
                line_dash="dash", 
                line_color=low_color,
                line_width=2,
                annotation_text=f"OR Low {day}: ${or_low:.2f}",
                annotation_position="bottom right",
                row=1
            )
            
            # Add vertical lines for OR start and end
            y_min = day_data['low'].min() * 0.998
            y_max = day_data['high'].max() * 1.002
            
            # OR Start line
            fig.add_shape(
                type="line",
                x0=or_start, x1=or_start,
                y0=y_min, y1=y_max,
                line=dict(color="blue", width=2, dash="solid"),
                row=1, col=1
            )
            
            # OR End line
            fig.add_shape(
                type="line",
                x0=or_end, x1=or_end,
                y0=y_min, y1=y_max,
                line=dict(color="purple", width=2, dash="solid"),
                row=1, col=1
            )
            
            # Add shaded box for OR time period
            fig.add_vrect(
                x0=or_start,
                x1=or_end,
                fillcolor="rgba(255, 255, 0, 0.1)",
                layer="below",
                line_width=0,
                annotation_text=f"OR {day}",
                annotation_position="top left",
                row=1
            )
            
            # Print OR analysis
            print(f"📊 Opening Range Analysis for {day}:")
            print(f"   OR High: ${or_high:.2f}")
            print(f"   OR Low: ${or_low:.2f}")
            print(f"   OR Size: ${or_high - or_low:.2f}")
            print(f"   OR Period: {or_start.strftime('%H:%M')} to {or_end.strftime('%H:%M')}")
            print(f"   Bars in OR: {len(or_data)}")
            print()
            
        except Exception as e:
            print(f"Could not analyze OR for {day}: {e}")
            continue
    
    # Update layout for better interactivity
    fig.update_layout(
        title=f'Interactive SPY Chart - {title_date}',
        height=800,
        xaxis_rangeslider_visible=False,
        hovermode='x unified',
        showlegend=True
    )
    
    # Update x-axis for better time display
    fig.update_xaxes(
        title="Time",
        tickformat="%H:%M" if end_date is None else "%m-%d %H:%M",
        row=2, col=1
    )
    
    # Update y-axes
    fig.update_yaxes(title="Price ($)", row=1, col=1)
    fig.update_yaxes(title="Volume", row=2, col=1)
    
    return fig

# Example usage:
# fig = plot_interactive_ohlc(spy_df, '2025-06-18')
# fig.show()

In [28]:
# Load and prepare the data first
spy_data = db.DBNStore.from_file('spy_ohlcv_20190102_20250710.dbn')
spy_df = spy_data.to_df()

# Apply timezone conversion to US/Eastern
import pytz
if spy_df.index.tz is not None:
    eastern = pytz.timezone('US/Eastern')
    spy_df.index = spy_df.index.tz_convert(eastern)

print(f"Data loaded: {len(spy_df)} bars")
print(f"Date range: {spy_df.index.min()} to {spy_df.index.max()}")
print(f"Timezone: {spy_df.index.tz}")

Data loaded: 1217772 bars
Date range: 2019-01-02 04:00:00-05:00 to 2025-07-09 19:56:00-04:00
Timezone: US/Eastern


In [30]:
# CRITICAL: Verify timezone handling for Opening Range calculations
print("🔍 TIMEZONE VERIFICATION:")
print(f"Data timezone: {spy_df.index.tz}")

# Check a sample day to verify the time structure
sample_day = '2025-06-18'
day_data = spy_df.loc[sample_day]

print(f"\n📅 Sample day: {sample_day}")
print(f"First bar: {day_data.index[0]} (Should be 4:00 AM Eastern)")
print(f"Last bar: {day_data.index[-1]} (Should be ~8:00 PM Eastern)")

# Check what 9:30 AM looks like in the data
market_open = day_data.between_time('09:30', '09:30').index[0] if len(day_data.between_time('09:30', '09:30')) > 0 else None
or_end = day_data.between_time('10:15', '10:15').index[0] if len(day_data.between_time('10:15', '10:15')) > 0 else None

print(f"\n🎯 OPENING RANGE VERIFICATION:")
print(f"Market open (9:30 AM): {market_open}")
print(f"OR end (10:15 AM): {or_end}")

# Show some bars around market open
print(f"\n📊 Bars around market open:")
market_open_area = day_data.between_time('09:25', '09:35')
for i, (timestamp, row) in enumerate(market_open_area.head(10).iterrows()):
    marker = "🎯" if timestamp.time() == pd.Timestamp('09:30:00').time() else "  "
    print(f"{marker} {timestamp}: O=${row['open']:.2f}, H=${row['high']:.2f}, L=${row['low']:.2f}, C=${row['close']:.2f}, V={row['volume']:,}")

print(f"\n✅ Data verification complete. Timezone appears correct for OR calculations.")

🔍 TIMEZONE VERIFICATION:
Data timezone: US/Eastern

📅 Sample day: 2025-06-18
First bar: 2025-06-18 04:00:00-04:00 (Should be 4:00 AM Eastern)
Last bar: 2025-06-18 19:56:00-04:00 (Should be ~8:00 PM Eastern)

🎯 OPENING RANGE VERIFICATION:
Market open (9:30 AM): 2025-06-18 09:30:00-04:00
OR end (10:15 AM): 2025-06-18 10:15:00-04:00

📊 Bars around market open:
   2025-06-18 09:25:00-04:00: O=$598.60, H=$598.65, L=$598.55, C=$598.59, V=2,420
   2025-06-18 09:26:00-04:00: O=$598.62, H=$598.66, L=$598.54, C=$598.55, V=1,732
   2025-06-18 09:27:00-04:00: O=$598.55, H=$598.67, L=$598.52, C=$598.58, V=1,770
   2025-06-18 09:28:00-04:00: O=$598.52, H=$598.52, L=$598.35, C=$598.47, V=2,576
   2025-06-18 09:29:00-04:00: O=$598.48, H=$598.53, L=$598.42, C=$598.44, V=11,902
🎯 2025-06-18 09:30:00-04:00: O=$598.43, H=$598.45, L=$597.64, C=$597.97, V=78,330
   2025-06-18 09:31:00-04:00: O=$597.96, H=$598.10, L=$597.65, C=$597.70, V=54,162
   2025-06-18 09:32:00-04:00: O=$597.71, H=$597.91, L=$597.64, C

In [32]:
spy_df.index.tz

<DstTzInfo 'US/Eastern' LMT-1 day, 19:04:00 STD>

In [31]:
# Example 1: Plot a single day with default OR times (9:30-10:15)
fig1 = plot_interactive_ohlc(spy_df, '2025-06-18')
fig1.show()

📊 Opening Range Analysis for 2025-06-18:
   OR High: $600.22
   OR Low: $597.36
   OR Size: $2.86
   OR Period: 09:30 to 10:15
   Bars in OR: 46



In [17]:
# Example 2: Plot with custom opening range times (9:30-10:00)
fig2 = plot_interactive_ohlc(spy_df, '2025-06-18', or_start_time="09:30:00", or_end_time="10:00:00")
fig2.show()

📊 Opening Range Analysis for 2025-06-18:
   OR High: $599.58
   OR Low: $597.36
   OR Size: $2.22
   OR Period: 09:30 to 10:00
   Bars in OR: 31



In [18]:
# Example 3: Plot multiple days (will show OR for each day)
fig3 = plot_interactive_ohlc(spy_df, '2025-06-18', '2025-06-20')
fig3.show()

📊 Opening Range Analysis for 2025-06-18:
   OR High: $600.22
   OR Low: $597.36
   OR Size: $2.86
   OR Period: 09:30 to 10:15
   Bars in OR: 46

📊 Opening Range Analysis for 2025-06-20:
   OR High: $599.46
   OR Low: $596.62
   OR Size: $2.84
   OR Period: 09:30 to 10:15
   Bars in OR: 46



# 🚨 CRITICAL TIMEZONE FIXES NEEDED

Based on the analysis above, there are several timezone issues in the backtesting codebase that need to be fixed:

## Issues Found:

1. **data_manager.py**: Only handles CSV files, but config uses DBN files
2. **main.py lines 81-87**: Converting dates to UTC instead of Eastern 
3. **Missing DBN + timezone support** in the backtesting pipeline

## Required Fixes:

1. Update `data_manager.py` to handle DBN files with Eastern timezone
2. Fix `main.py` timezone filtering to use Eastern instead of UTC
3. Ensure backtester passes Eastern timezone to Cerebro
4. Verify all OR calculations use consistent Eastern time

## Current Status:
✅ Notebook data loading: CORRECT (Eastern timezone)  
❌ Backtester data loading: NEEDS FIX (CSV only, no timezone handling)  
❌ Date filtering in main.py: NEEDS FIX (using UTC instead of Eastern)  

In [22]:
# 🔍 INVESTIGATE OR DISCREPANCY FOR JUNE 18TH
print("🚨 DEBUGGING OR DISCREPANCY - JUNE 18TH")
print("=" * 60)

# Check full dataset first
june_18_full = spy_df.loc['2025-06-18']
print(f"📅 Full dataset for 2025-06-18:")
print(f"   Total bars: {len(june_18_full)}")
print(f"   First bar: {june_18_full.index[0]}")
print(f"   Last bar: {june_18_full.index[-1]}")

# Get OR period for full dataset
or_start_full = pd.Timestamp('2025-06-18 09:30:00', tz=june_18_full.index.tz)
or_end_full = pd.Timestamp('2025-06-18 10:15:00', tz=june_18_full.index.tz)
or_data_full = june_18_full.loc[or_start_full:or_end_full]

print(f"\n📊 OR Analysis (FULL DATASET):")
print(f"   OR Period: {or_start_full} to {or_end_full}")
print(f"   OR Bars: {len(or_data_full)}")
if not or_data_full.empty:
    or_high_full = or_data_full['high'].max()
    or_low_full = or_data_full['low'].min()
    print(f"   OR High: ${or_high_full:.2f}")
    print(f"   OR Low: ${or_low_full:.2f}")
    print(f"   OR Size: ${or_high_full - or_low_full:.2f}")

# Now simulate what the backtester sees (test split)
print(f"\n" + "="*60)
print("🎯 SIMULATING BACKTESTER DATA (TEST SPLIT)")

# Apply same split ratio as backtester
split_ratio = 0.8  # 80% train, 20% test
split_index = int(len(spy_df) * split_ratio)
test_split_df = spy_df.iloc[split_index:]

print(f"📊 Test split info:")
print(f"   Split ratio: {split_ratio} (80% train, 20% test)")
print(f"   Test split starts: {test_split_df.index[0]}")
print(f"   Test split ends: {test_split_df.index[-1]}")

# Check if June 18th is in test split
try:
    june_18_test = test_split_df.loc['2025-06-18']
    print(f"\n✅ June 18th found in test split:")
    print(f"   Test split bars for June 18th: {len(june_18_test)}")
    print(f"   First bar: {june_18_test.index[0]}")
    print(f"   Last bar: {june_18_test.index[-1]}")
    
    # Get OR for test split
    or_data_test = june_18_test.loc[or_start_full:or_end_full]
    print(f"\n📊 OR Analysis (TEST SPLIT - what backtester sees):")
    print(f"   OR Bars in test split: {len(or_data_test)}")
    
    if not or_data_test.empty:
        or_high_test = or_data_test['high'].max()
        or_low_test = or_data_test['low'].min()
        print(f"   OR High: ${or_high_test:.2f}")
        print(f"   OR Low: ${or_low_test:.2f}")
        print(f"   OR Size: ${or_high_test - or_low_test:.2f}")
        
        # Compare values
        print(f"\n🔍 COMPARISON:")
        print(f"   Full dataset OR High: ${or_high_full:.2f}")
        print(f"   Test split OR High:   ${or_high_test:.2f}")
        print(f"   Difference: ${abs(or_high_full - or_high_test):.2f}")
        print(f"   ")
        print(f"   Full dataset OR Low:  ${or_low_full:.2f}")
        print(f"   Test split OR Low:    ${or_low_test:.2f}")
        print(f"   Difference: ${abs(or_low_full - or_low_test):.2f}")
        
        if or_high_full != or_high_test or or_low_full != or_low_test:
            print(f"\n🚨 DISCREPANCY FOUND!")
            print(f"   The test split has different OR values than the full dataset")
            print(f"   This explains why backtester shows different values")
    else:
        print(f"   ❌ No OR data found in test split!")
        
except KeyError:
    print(f"\n❌ June 18th NOT found in test split!")
    print(f"   This means the test split starts after June 18th")
    print(f"   Backtester cannot analyze June 18th OR")

print(f"\n" + "="*60)
print("🎯 BACKTESTER OUTPUT REFERENCE:")
print("   Backtester showed: OR High=$599.74, Low=$598.76")
print("   This should match the test split values above")

🚨 DEBUGGING OR DISCREPANCY - JUNE 18TH
📅 Full dataset for 2025-06-18:
   Total bars: 788
   First bar: 2025-06-18 04:00:00-04:00
   Last bar: 2025-06-18 19:56:00-04:00

📊 OR Analysis (FULL DATASET):
   OR Period: 2025-06-18 09:30:00-04:00 to 2025-06-18 10:15:00-04:00
   OR Bars: 46
   OR High: $600.22
   OR Low: $597.36
   OR Size: $2.86

🎯 SIMULATING BACKTESTER DATA (TEST SPLIT)
📊 Test split info:
   Split ratio: 0.8 (80% train, 20% test)
   Test split starts: 2024-03-21 10:18:00-04:00
   Test split ends: 2025-07-09 19:56:00-04:00

✅ June 18th found in test split:
   Test split bars for June 18th: 788
   First bar: 2025-06-18 04:00:00-04:00
   Last bar: 2025-06-18 19:56:00-04:00

📊 OR Analysis (TEST SPLIT - what backtester sees):
   OR Bars in test split: 46
   OR High: $600.22
   OR Low: $597.36
   OR Size: $2.86

🔍 COMPARISON:
   Full dataset OR High: $600.22
   Test split OR High:   $600.22
   Difference: $0.00
   
   Full dataset OR Low:  $597.36
   Test split OR Low:    $597.36
 

In [23]:
# 🔬 DETAILED ANALYSIS: What bars is the backtester actually seeing?
print("🔬 DETAILED OR BAR ANALYSIS")
print("=" * 80)

# Get the exact OR period data
june_18_data = spy_df.loc['2025-06-18']
or_start = pd.Timestamp('2025-06-18 09:30:00', tz=june_18_data.index.tz)
or_end = pd.Timestamp('2025-06-18 10:15:00', tz=june_18_data.index.tz)
or_bars = june_18_data.loc[or_start:or_end]

print(f"📊 Opening Range Period: {or_start} to {or_end}")
print(f"📊 Total OR bars: {len(or_bars)}")
print(f"📊 Actual OR High: ${or_bars['high'].max():.2f}")
print(f"📊 Actual OR Low: ${or_bars['low'].min():.2f}")

print(f"\n🔍 First 10 OR bars:")
for i, (timestamp, row) in enumerate(or_bars.head(10).iterrows()):
    print(f"   {i+1:2d}. {timestamp} | O=${row['open']:.2f} H=${row['high']:.2f} L=${row['low']:.2f} C=${row['close']:.2f}")

print(f"\n🔍 Last 10 OR bars:")
for i, (timestamp, row) in enumerate(or_bars.tail(10).iterrows()):
    total_idx = len(or_bars) - 10 + i + 1
    print(f"   {total_idx:2d}. {timestamp} | O=${row['open']:.2f} H=${row['high']:.2f} L=${row['low']:.2f} C=${row['close']:.2f}")

# Find the highest and lowest bars
max_bar = or_bars.loc[or_bars['high'].idxmax()]
min_bar = or_bars.loc[or_bars['low'].idxmin()]

print(f"\n🎯 HIGHEST BAR in OR:")
print(f"   Time: {max_bar.name}")
print(f"   OHLC: O=${max_bar['open']:.2f} H=${max_bar['high']:.2f} L=${max_bar['low']:.2f} C=${max_bar['close']:.2f}")

print(f"\n🎯 LOWEST BAR in OR:")
print(f"   Time: {min_bar.name}")
print(f"   OHLC: O=${min_bar['open']:.2f} H=${min_bar['high']:.2f} L=${min_bar['low']:.2f} C=${min_bar['close']:.2f}")

# Check if backtester might be using UTC timestamps
print(f"\n" + "="*80)
print("🚨 POTENTIAL TIMEZONE ISSUE CHECK")

# Convert to UTC to see if that matches backtester values
or_start_utc = or_start.tz_convert('UTC')
or_end_utc = or_end.tz_convert('UTC')

print(f"🕐 OR period in Eastern: {or_start} to {or_end}")
print(f"🕐 OR period in UTC: {or_start_utc} to {or_end_utc}")

# Check if there's data that would give us the backtester's values
print(f"\n🔍 Searching for data that would give OR High=$599.74, Low=$598.76...")

# Look for bars around those price levels
high_target = 599.74
low_target = 598.76

matching_high_bars = june_18_data[abs(june_18_data['high'] - high_target) < 0.01]
matching_low_bars = june_18_data[abs(june_18_data['low'] - low_target) < 0.01]

print(f"\n🎯 Bars with high ≈ ${high_target}:")
for timestamp, row in matching_high_bars.iterrows():
    print(f"   {timestamp} | O=${row['open']:.2f} H=${row['high']:.2f} L=${row['low']:.2f} C=${row['close']:.2f}")

print(f"\n🎯 Bars with low ≈ ${low_target}:")
for timestamp, row in matching_low_bars.iterrows():
    print(f"   {timestamp} | O=${row['open']:.2f} H=${row['high']:.2f} L=${row['low']:.2f} C=${row['close']:.2f}")

# Check if the backtester might be using a different time range
print(f"\n" + "="*80)
print("🔍 HYPOTHESIS: Backtester using wrong time boundaries")

# Test different potential time ranges that might give those values
test_ranges = [
    ('09:30:00', '10:14:00'),  # Missing last minute
    ('09:31:00', '10:15:00'),  # Missing first minute  
    ('09:32:00', '10:16:00'),  # Shifted forward
    ('09:29:00', '10:14:00'),  # Different boundaries
]

for start_time, end_time in test_ranges:
    test_start = pd.Timestamp(f'2025-06-18 {start_time}', tz=june_18_data.index.tz)
    test_end = pd.Timestamp(f'2025-06-18 {end_time}', tz=june_18_data.index.tz)
    
    try:
        test_or_data = june_18_data.loc[test_start:test_end]
        if not test_or_data.empty:
            test_high = test_or_data['high'].max()
            test_low = test_or_data['low'].min()
            
            if abs(test_high - 599.74) < 0.1 or abs(test_low - 598.76) < 0.1:
                print(f"🎯 POTENTIAL MATCH: {start_time} to {end_time}")
                print(f"   High: ${test_high:.2f} (target: $599.74)")
                print(f"   Low: ${test_low:.2f} (target: $598.76)")
                print(f"   Bars: {len(test_or_data)}")
    except:
        continue

🔬 DETAILED OR BAR ANALYSIS
📊 Opening Range Period: 2025-06-18 09:30:00-04:00 to 2025-06-18 10:15:00-04:00
📊 Total OR bars: 46
📊 Actual OR High: $600.22
📊 Actual OR Low: $597.36

🔍 First 10 OR bars:
    1. 2025-06-18 09:30:00-04:00 | O=$598.43 H=$598.45 L=$597.64 C=$597.97
    2. 2025-06-18 09:31:00-04:00 | O=$597.96 H=$598.10 L=$597.65 C=$597.70
    3. 2025-06-18 09:32:00-04:00 | O=$597.71 H=$597.91 L=$597.64 C=$597.69
    4. 2025-06-18 09:33:00-04:00 | O=$597.69 H=$597.74 L=$597.47 C=$597.63
    5. 2025-06-18 09:34:00-04:00 | O=$597.62 H=$597.62 L=$597.36 C=$597.50
    6. 2025-06-18 09:35:00-04:00 | O=$597.52 H=$597.98 L=$597.50 C=$597.97
    7. 2025-06-18 09:36:00-04:00 | O=$597.97 H=$597.97 L=$597.64 C=$597.84
    8. 2025-06-18 09:37:00-04:00 | O=$597.83 H=$598.20 L=$597.78 C=$598.04
    9. 2025-06-18 09:38:00-04:00 | O=$598.05 H=$598.24 L=$597.99 C=$598.04
   10. 2025-06-18 09:39:00-04:00 | O=$598.05 H=$598.17 L=$598.02 C=$598.11

🔍 Last 10 OR bars:
   37. 2025-06-18 10:06:00-04:00

In [24]:
# 🕘 CHECK BAR TIMING AROUND MARKET OPEN
print("🕘 CHECKING BAR AVAILABILITY AROUND MARKET OPEN")
print("=" * 70)

june_18_data = spy_df.loc['2025-06-18']

# Check bars around market open time
market_open_window = june_18_data.between_time('09:25', '09:35')

print(f"📊 Bars around market open (9:25 - 9:35 AM):")
for timestamp, row in market_open_window.iterrows():
    time_str = timestamp.strftime('%H:%M:%S')
    marker = "🎯" if time_str == "09:30:00" else "  "
    print(f"{marker} {timestamp} | O=${row['open']:.2f} H=${row['high']:.2f} L=${row['low']:.2f} C=${row['close']:.2f} V={row['volume']:,}")

# Check if 9:30 bar exists
has_930_bar = any(t.time() == pd.Timestamp('09:30:00').time() for t in market_open_window.index)
has_931_bar = any(t.time() == pd.Timestamp('09:31:00').time() for t in market_open_window.index)

print(f"\n🔍 Bar availability check:")
print(f"   9:30:00 bar exists: {'✅ YES' if has_930_bar else '❌ NO'}")
print(f"   9:31:00 bar exists: {'✅ YES' if has_931_bar else '❌ NO'}")

if not has_930_bar:
    print(f"\n🚨 ISSUE FOUND: No 9:30:00 bar!")
    print(f"   This explains why backtester starts OR calculation at 9:32!")
    print(f"   The data might be missing the exact market open minute.")
    
    # Find the first available bar in the OR window
    or_window = june_18_data.between_time('09:30', '10:15')
    first_or_bar = or_window.index[0]
    print(f"   First available OR bar: {first_or_bar}")
    print(f"   This matches backtester log: 'OR Calculation STARTED at {first_or_bar.strftime('%H:%M')}'")

# Check the specific values that backtester reported
print(f"\n🎯 BACKTESTER VALUES VERIFICATION:")
print(f"Backtester reported first bar: O=599.68, H=599.74, L=599.68, C=599.72")

# Find bars that match these values
matching_bars = june_18_data[
    (abs(june_18_data['open'] - 599.68) < 0.01) & 
    (abs(june_18_data['high'] - 599.74) < 0.01) &
    (abs(june_18_data['low'] - 599.68) < 0.01) &
    (abs(june_18_data['close'] - 599.72) < 0.01)
]

print(f"\nBars matching backtester's first bar values:")
for timestamp, row in matching_bars.iterrows():
    print(f"   {timestamp} | O=${row['open']:.2f} H=${row['high']:.2f} L=${row['low']:.2f} C=${row['close']:.2f}")

# Calculate OR using only the bars that backtester would see
print(f"\n" + "="*70)
print("🧮 RECALCULATING OR USING BACKTESTER'S LOGIC")

# Simulate backtester behavior: start from first available bar in OR window
or_window_full = june_18_data.between_time('09:30', '10:15')
if not or_window_full.empty:
    # This is what the backtester actually sees
    backtester_or_high = or_window_full['high'].max()
    backtester_or_low = or_window_full['low'].min()
    
    print(f"Backtester should see:")
    print(f"   OR High: ${backtester_or_high:.2f}")
    print(f"   OR Low: ${backtester_or_low:.2f}")
    print(f"   OR Size: ${backtester_or_high - backtester_or_low:.2f}")
    
    print(f"\nActual backtester output:")
    print(f"   OR High: $599.74")
    print(f"   OR Low: $598.76")
    print(f"   OR Size: ${599.74 - 598.76:.2f}")
    
    if abs(backtester_or_high - 599.74) > 0.1 or abs(backtester_or_low - 598.76) > 0.1:
        print(f"\n🚨 STILL A DISCREPANCY!")
        print(f"   Even accounting for missing bars, values don't match")
        print(f"   There must be another issue...")
    else:
        print(f"\n✅ VALUES MATCH!")
        print(f"   The discrepancy is explained by missing 9:30 minute bar")

🕘 CHECKING BAR AVAILABILITY AROUND MARKET OPEN
📊 Bars around market open (9:25 - 9:35 AM):
   2025-06-18 09:25:00-04:00 | O=$598.60 H=$598.65 L=$598.55 C=$598.59 V=2,420
   2025-06-18 09:26:00-04:00 | O=$598.62 H=$598.66 L=$598.54 C=$598.55 V=1,732
   2025-06-18 09:27:00-04:00 | O=$598.55 H=$598.67 L=$598.52 C=$598.58 V=1,770
   2025-06-18 09:28:00-04:00 | O=$598.52 H=$598.52 L=$598.35 C=$598.47 V=2,576
   2025-06-18 09:29:00-04:00 | O=$598.48 H=$598.53 L=$598.42 C=$598.44 V=11,902
🎯 2025-06-18 09:30:00-04:00 | O=$598.43 H=$598.45 L=$597.64 C=$597.97 V=78,330
   2025-06-18 09:31:00-04:00 | O=$597.96 H=$598.10 L=$597.65 C=$597.70 V=54,162
   2025-06-18 09:32:00-04:00 | O=$597.71 H=$597.91 L=$597.64 C=$597.69 V=43,318
   2025-06-18 09:33:00-04:00 | O=$597.69 H=$597.74 L=$597.47 C=$597.63 V=33,837
   2025-06-18 09:34:00-04:00 | O=$597.62 H=$597.62 L=$597.36 C=$597.50 V=35,644
   2025-06-18 09:35:00-04:00 | O=$597.52 H=$597.98 L=$597.50 C=$597.97 V=43,272

🔍 Bar availability check:
   9:30

In [26]:
# 🔍 TEST THE DATA_MANAGER DIRECTLY
print("🔍 TESTING DATA_MANAGER.PY DIRECTLY")
print("=" * 60)

# Let's simulate what the backtester is doing
import sys
import os
sys.path.append(r'C:\Users\Tom\workspace\trading_bot')

from common.data_manager import load_ohlc_data

# Load data exactly like the backtester does
dbn_file_path = r'C:\Users\Tom\workspace\trading_bot\data\spy_ohlcv_20190102_20250710.dbn'
loaded_df = load_ohlc_data(dbn_file_path)

print(f"Data loaded from data_manager.py:")
print(f"   Shape: {loaded_df.shape}")
print(f"   Index timezone: {loaded_df.index.tz}")
print(f"   First timestamp: {loaded_df.index[0]}")
print(f"   Last timestamp: {loaded_df.index[-1]}")

# Check June 18th specifically
june_18_dm = loaded_df.loc['2025-06-18']
print(f"\nJune 18th data from data_manager:")
print(f"   Total bars: {len(june_18_dm)}")
print(f"   First bar: {june_18_dm.index[0]}")
print(f"   Sample OHLC: O=${june_18_dm.iloc[0]['open']:.2f}, H=${june_18_dm.iloc[0]['high']:.2f}, L=${june_18_dm.iloc[0]['low']:.2f}, C=${june_18_dm.iloc[0]['close']:.2f}")

# Check what OR would be from data_manager
or_start_dm = pd.Timestamp('2025-06-18 09:30:00', tz=june_18_dm.index.tz)
or_end_dm = pd.Timestamp('2025-06-18 10:15:00', tz=june_18_dm.index.tz)
or_data_dm = june_18_dm.loc[or_start_dm:or_end_dm]

if not or_data_dm.empty:
    or_high_dm = or_data_dm['high'].max()
    or_low_dm = or_data_dm['low'].min()
    print(f"\nOR from data_manager.py:")
    print(f"   OR High: ${or_high_dm:.2f}")
    print(f"   OR Low: ${or_low_dm:.2f}")
    print(f"   OR Size: ${or_high_dm - or_low_dm:.2f}")
    print(f"   OR Bars: {len(or_data_dm)}")
    
    # Check the 9:32 bar specifically
    try:
        bar_932 = june_18_dm.loc[pd.Timestamp('2025-06-18 09:32:00', tz=june_18_dm.index.tz)]
        print(f"\n9:32 AM bar from data_manager:")
        print(f"   O=${bar_932['open']:.2f}, H=${bar_932['high']:.2f}, L=${bar_932['low']:.2f}, C=${bar_932['close']:.2f}")
        
        if (abs(bar_932['open'] - 599.68) < 0.01 and 
            abs(bar_932['high'] - 599.74) < 0.01 and
            abs(bar_932['low'] - 599.68) < 0.01 and
            abs(bar_932['close'] - 599.72) < 0.01):
            print(f"   🚨 THIS MATCHES THE BACKTESTER'S INCORRECT VALUES!")
            print(f"   The data_manager is returning wrong data!")
        else:
            print(f"   ✅ This does NOT match backtester values (good)")
            print(f"   Expected from backtester: O=599.68, H=599.74, L=599.68, C=599.72")
            
    except KeyError:
        print(f"\n❌ No 9:32 AM bar found in data_manager data")

else:
    print(f"\n❌ No OR data found for June 18th in data_manager")

print(f"\n" + "="*60)
print("🎯 CONCLUSION:")
print("If the data_manager shows UTC timezone, then the bug is in data_manager.py")
print("If it shows Eastern timezone, then the bug is in the feature engineering or main.py")

🔍 TESTING DATA_MANAGER.PY DIRECTLY
Data loaded from data_manager.py:
   Shape: (1217772, 9)
   Index timezone: UTC
   First timestamp: 2019-01-02 09:00:00+00:00
   Last timestamp: 2025-07-09 23:56:00+00:00

June 18th data from data_manager:
   Total bars: 788
   First bar: 2025-06-18 08:00:00+00:00
   Sample OHLC: O=$599.00, H=$599.00, L=$598.97, C=$598.98

OR from data_manager.py:
   OR High: $599.74
   OR Low: $598.76
   OR Size: $0.98
   OR Bars: 30

9:32 AM bar from data_manager:
   O=$599.68, H=$599.74, L=$599.68, C=$599.72
   🚨 THIS MATCHES THE BACKTESTER'S INCORRECT VALUES!
   The data_manager is returning wrong data!

🎯 CONCLUSION:
If the data_manager shows UTC timezone, then the bug is in data_manager.py
If it shows Eastern timezone, then the bug is in the feature engineering or main.py
